# Clase 3 — Plataforma de ML: MLflow

**Objetivo:** registrar experimentos (parámetros, métricas, gráficos, modelos), comparar runs y usar
el **Model Registry** con alias `champion` / `challenger`, que es lo que usan los DAGs del TP.

Para practicar sin Docker usamos un MLflow **local** (una base SQLite en `clases/_salidas/`). La API
es idéntica a la del servidor del docker-compose: solo cambia la URI.

| Componente | Local (esta clase) | Docker (TP) |
|---|---|---|
| *Backend store* (runs, params, métricas) | SQLite | PostgreSQL (`mlflow_db`) |
| *Artifact store* (modelos, gráficos) | carpeta local | MinIO, bucket `s3://mlflow` |
| URI | `sqlite:///…/mlflow_clase3.db` | `http://localhost:5001` |

In [1]:
from helpers import OUT, check, cuestionario, quiz, safe_check, tarjetas

import os
import warnings

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from ipywidgets import interact, widgets
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score

from burnout.config import BEST_PARAMS_AMQ1, LABELS, SEED, SKOPS_TRUSTED_TYPES
from burnout.data import load_raw, make_split
from burnout.model import make_pipeline, test_metrics

TRACKING_URI = f"sqlite:///{(OUT / 'mlflow_clase3.db').as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)
experimento = mlflow.set_experiment("clase3_burnout")
print("MLflow:", TRACKING_URI)

X_train, X_test, y_train, y_test = make_split(load_raw())
# Para que la clase sea ágil, entrenamos con una muestra de 10.000 filas
muestra = X_train.sample(10_000, random_state=SEED).index
Xs, ys = X_train.loc[muestra], y_train.loc[muestra]

2026/11/11 14:39:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/11 14:39:13 INFO mlflow.store.db.utils: Updating database tables


2026/11/11 14:39:14 INFO mlflow.tracking.fluent: Experiment with name 'clase3_burnout' does not exist. Creating a new experiment.


MLflow: sqlite:///C:/Users/USUARIO/CEIA/MLOps-TP-AVS/clases/_salidas/mlflow_clase3.db


## 📖 Teoría: por qué registrar experimentos

En AMq1 probamos 6 familias de modelos y 40 combinaciones de hiperparámetros. ¿Podrías decir hoy,
sin volver a correr nada, **qué parámetros tuvo la prueba 17 y cuánto dio**? Sin un registro, esa
información se pierde en celdas sobrescritas y planillas sueltas.

Una plataforma de ML resuelve tres problemas:

1. **Seguimiento (*tracking*):** cada entrenamiento queda guardado con sus parámetros, métricas,
   código y archivos.
2. **Comparación:** ver lado a lado cientos de runs y entender qué funcionó.
3. **Gestión del ciclo de vida:** qué modelo está en producción, cuál es candidato, cómo volver atrás.

**Componentes de MLflow:**

```mermaid
flowchart LR
    subgraph Cliente["Tu código (notebook, DAG)"]
        L["mlflow.log_param / log_metric<br>log_model"]
    end
    L -- "HTTP" --> S["Servidor de MLflow<br>(tracking server)"]
    S --> B[("Backend store<br>PostgreSQL<br>runs · params · métricas")]
    S --> A[("Artifact store<br>MinIO s3://mlflow<br>modelos · figuras")]
    S --> R["Model Registry<br>versiones + alias"]
```

**Jerarquía de conceptos:**

```mermaid
flowchart TD
    E["Experimento: Burnout Risk"] --> P["Run padre: optuna_search"]
    P --> H1["Run hijo: trial_00"]
    P --> H2["Run hijo: trial_01"]
    P --> H3["… 40 pruebas"]
    E --> F["Run: final_from_optuna"]
    F -- "log_model + register" --> V["burnout_model_prod v1"]
    V -. "alias" .-> CH(["@champion"])
```

| Qué se registra | Ejemplo en el TP | Método |
|---|---|---|
| Parámetro | `max_depth = 2` | `mlflow.log_param(s)` |
| Métrica | `test_f1_macro = 0.536` | `mlflow.log_metric(s)` |
| Etiqueta | `stage = optuna` | `tags=` / `mlflow.set_tag` |
| Artefacto | `optuna_historia.png` | `mlflow.log_figure` / `log_artifact` |
| Modelo | el Pipeline completo | `mlflow.sklearn.log_model` |
| Dataset | el split de MinIO | `mlflow.log_input` |

**Model Registry y alias.** Registrar crea **versiones numeradas** (v1, v2…) de un modelo con nombre.
Un **alias** es una etiqueta móvil: producción siempre pide `models:/burnout_model_prod@champion`, y
**promover** un modelo es mover el alias, sin tocar el código que lo usa. Volver atrás (*rollback*)
es moverlo de nuevo.

## 1. Conceptos
- **Experimento**: agrupa runs de un mismo problema (en el TP: *Burnout Risk*).
- **Run**: una ejecución. Guarda **parámetros** (entradas), **métricas** (resultados),
  **tags** (etiquetas libres) y **artefactos** (archivos: modelos, gráficos, datasets).
- **Modelo registrado**: un nombre (`burnout_model_prod`) con **versiones** (1, 2, 3…) y
  **alias** que apuntan a una versión (`champion`, `challenger`).

## 2. Registro manual de un run

In [2]:
params = {"max_depth": 3, "learning_rate": 0.1, "n_estimators": 150}

with mlflow.start_run(run_name="manual_xgb", tags={"autor": "clase3"}) as run:
    modelo = make_pipeline(**params).fit(Xs, ys)
    metricas = test_metrics(y_test, modelo.predict(X_test))

    mlflow.log_params(params)
    mlflow.log_metrics(metricas)

    fig, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test, display_labels=LABELS, ax=ax, colorbar=False)
    mlflow.log_figure(fig, "matriz_confusion.png")
    plt.close(fig)

print(f"run_id: {run.info.run_id}")
pd.Series(metricas).round(4)

run_id: 66b9398483e9419eb59b26cebf3a67d8


f1_macro             0.5268
balanced_accuracy    0.5140
recall_low           0.5379
recall_medium        0.5852
recall_high          0.4189
dtype: float64

### ✏️ Ejercicio 1 — registrá una métrica nueva
Abrí un run llamado `"ejercicio1"` y registrá la métrica `"f1_train"` con el F1 macro **en train**
(`Xs`, `ys`) del `modelo` de arriba. Usá `f1_score(..., average="macro")`.

In [3]:
from sklearn.metrics import f1_score

# ← tu código acá

In [4]:
def _ej1():
    runs = mlflow.search_runs(filter_string="attributes.run_name = 'ejercicio1'")
    return not runs.empty and "metrics.f1_train" in runs and runs["metrics.f1_train"].notna().any()


safe_check(
    "Ejercicio 1",
    _ej1,
    "`with mlflow.start_run(run_name='ejercicio1'):` y adentro "
    "`mlflow.log_metric('f1_train', f1_score(ys, modelo.predict(Xs), average='macro'))`.",
)

⏳ **Ejercicio 1**: todavía no. *Pista:* `with mlflow.start_run(run_name='ejercicio1'):` y adentro `mlflow.log_metric('f1_train', f1_score(ys, modelo.predict(Xs), average='macro'))`.

False

<details><summary>Ver solución</summary>

```python
with mlflow.start_run(run_name="ejercicio1"):
    mlflow.log_metric("f1_train", f1_score(ys, modelo.predict(Xs), average="macro"))
```
</details>

## 3. 🎛️ Tu propio tablero de experimentos
Cada vez que tocás **Entrenar y registrar** se crea un run nuevo. La tabla de abajo se arma con
`mlflow.search_runs`, igual que la que vas a ver en la UI.

In [5]:
@interact(
    max_depth=widgets.IntSlider(min=1, max=10, value=4),
    learning_rate=widgets.FloatLogSlider(base=10, min=-2, max=-0.5, value=0.1),
    n_estimators=widgets.IntSlider(min=50, max=400, step=50, value=150),
    manual=True,
    manual_name="Entrenar y registrar",
)
def entrenar_y_registrar(max_depth, learning_rate, n_estimators):
    p = {"max_depth": max_depth, "learning_rate": round(learning_rate, 4), "n_estimators": n_estimators}
    with mlflow.start_run(run_name="tablero", tags={"autor": "clase3"}):
        m = make_pipeline(**p).fit(Xs, ys)
        mlflow.log_params(p)
        mlflow.log_metrics(test_metrics(y_test, m.predict(X_test)))
    runs = mlflow.search_runs(filter_string="attributes.run_name = 'tablero'",
                              order_by=["metrics.f1_macro DESC"])
    cols = ["params.max_depth", "params.learning_rate", "params.n_estimators", "metrics.f1_macro"]
    print(runs[cols].head(10).round(4).to_string(index=False))

interactive(children=(IntSlider(value=4, description='max_depth', max=10, min=1), FloatLogSlider(value=0.1, de…

## 4. Autolog
`mlflow.sklearn.autolog()` registra solo los parámetros, métricas de entrenamiento y el modelo de
cualquier `.fit()` de scikit-learn. Cómodo, pero registra **mucho**: para producción preferimos
registrar a mano lo que importa.

In [6]:
mlflow.sklearn.autolog(log_models=False)
with mlflow.start_run(run_name="autolog"):
    make_pipeline(max_depth=3).fit(Xs, ys)
mlflow.sklearn.autolog(disable=True)

ultimo = mlflow.search_runs(filter_string="attributes.run_name = 'autolog'").iloc[0]
print(f"Autolog registró {sum(c.startswith('params.') for c in ultimo.index if pd.notna(ultimo[c]))} parámetros, por ejemplo:")
print(ultimo.filter(like="params.xgb__").dropna().head(8).to_string())

C:\Users\USUARIO\CEIA\MLOps-TP-AVS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


Autolog registró 72 parámetros, por ejemplo:
params.xgb__min_child_weight         None
params.xgb__colsample_bynode         None
params.xgb__eval_metric          mlogloss
params.xgb__colsample_bylevel        None
params.xgb__random_state               42
params.xgb__max_cat_to_onehot        None
params.xgb__num_class                   3
params.xgb__importance_type          None


## 5. Runs anidados: una búsqueda de hiperparámetros
Es el patrón del notebook `notebooks/experimento_mlflow_optuna.ipynb` del TP: un run padre y un run
hijo por prueba (`nested=True`). Acá con 6 pruebas y 3 folds para que sea rápido.

In [7]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
cv3 = StratifiedKFold(3, shuffle=True, random_state=SEED)


def objetivo(trial):
    p = {
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.3, log=True),
        "n_estimators": 150,
    }
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        score = cross_val_score(make_pipeline(**p), Xs, ys, cv=cv3, scoring="f1_macro").mean()
        mlflow.log_params(p)
        mlflow.log_metric("cv_f1_macro", score)
    return score


with mlflow.start_run(run_name="busqueda_clase3") as padre:
    estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    estudio.optimize(objetivo, n_trials=6)
    mlflow.log_metric("best_cv_f1_macro", estudio.best_value)

hijos = mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{padre.info.run_id}'")
hijos[["tags.mlflow.runName", "params.max_depth", "params.learning_rate", "metrics.cv_f1_macro"]].round(4)

,tags.mlflow.runName,params.max_depth,params.learning_rate,metrics.cv_f1_macro
0,trial_5,2,0.27991818073277003,0.5163
1,trial_4,5,0.15317709628184267,0.5017
2,trial_3,2,0.22044355215811345,0.5187
3,trial_2,2,0.04296509486733454,0.5194
4,trial_1,5,0.11906379991333112,0.5088
5,trial_0,3,0.2678154091306088,0.5096


## 6. Model Registry: champion y challenger
Registramos dos versiones del modelo: el de AMq1 y uno "experimental". El alias `champion` marca la
versión que está en producción; los DAGs y la API cargan siempre `models:/<nombre>@champion`,
así **promover un modelo no requiere tocar código**.

> **Detalle de MLflow 3:** guarda los modelos de scikit-learn con **skops**, un formato que (a
> diferencia de pickle) no ejecuta código arbitrario al cargar y solo acepta tipos declarados como
> confiables. XGBoost no está en su lista, así que lo declaramos con `skops_trusted_types`
> (`burnout.config.SKOPS_TRUSTED_TYPES`). Sin eso, `log_model` falla.

In [8]:
from mlflow.models import infer_signature

NOMBRE = "burnout_clase3"
client = mlflow.MlflowClient()


def registrar(params: dict, run_name: str) -> str:
    with mlflow.start_run(run_name=run_name):
        m = make_pipeline(**params).fit(Xs, ys)
        mlflow.log_metrics(test_metrics(y_test, m.predict(X_test)))
        info = mlflow.sklearn.log_model(m, name="model", registered_model_name=NOMBRE,
                                        signature=infer_signature(Xs, m.predict(Xs)),
                                        skops_trusted_types=SKOPS_TRUSTED_TYPES)
    return str(info.registered_model_version)


v_amq1 = registrar(BEST_PARAMS_AMQ1, "amq1")
v_exp = registrar({"max_depth": 8, "n_estimators": 300}, "experimental")
client.set_registered_model_alias(NOMBRE, "champion", v_amq1)
print(f"Versiones: AMq1 = v{v_amq1}, experimental = v{v_exp}. Champion → v{v_amq1}")

Successfully registered model 'burnout_clase3'.
Created version '1' of model 'burnout_clase3'.


Versiones: AMq1 = v1, experimental = v2. Champion → v1


Registered model 'burnout_clase3' already exists. Creating a new version of this model...
Created version '2' of model 'burnout_clase3'.


In [9]:
champion = mlflow.sklearn.load_model(f"models:/{NOMBRE}@champion")
print("Predicciones del champion:", [LABELS[i] for i in champion.predict(X_test.head(5))])

Predicciones del champion: ['Medium', 'High', 'Medium', 'High', 'Low']


### ✏️ Ejercicio 2 — ¿desafiante o campeón?
1. Poné el alias `"challenger"` a la versión experimental (`v_exp`).
2. Compará el F1 macro en test del champion y del challenger (cargalos por alias) y guardalos en un
   diccionario `f1 = {"champion": ..., "challenger": ...}`.
3. Si el challenger es mejor, movele el alias `"champion"`. Si no, borrá el alias `"challenger"`.

Es exactamente la lógica del DAG de reentrenamiento que haremos en la segunda entrega.

In [10]:
# ← tu código acá

In [11]:
def _ej2():
    # MLflow devuelve la versión del alias como int; registrar() la devuelve como texto
    aliases = {a: str(v) for a, v in client.get_registered_model(NOMBRE).aliases.items()}
    comparados = set(globals().get("f1", {})) == {"champion", "challenger"}  # hiciste la comparación
    f1_version = {
        v: test_metrics(y_test, mlflow.sklearn.load_model(f"models:/{NOMBRE}/{v}").predict(X_test))["f1_macro"]
        for v in (v_amq1, v_exp)
    }
    mejor = max(f1_version, key=f1_version.get)
    return comparados and aliases.get("champion") == mejor and "challenger" not in aliases


safe_check(
    "Ejercicio 2",
    _ej2,
    "`client.set_registered_model_alias(NOMBRE, 'challenger', v_exp)`; cargá ambos con "
    "`mlflow.sklearn.load_model(f'models:/{NOMBRE}@challenger')` y guardá los F1 en `f1`; al final "
    "`client.delete_registered_model_alias(NOMBRE, 'challenger')`.",
)

⏳ **Ejercicio 2**: todavía no. *Pista:* `client.set_registered_model_alias(NOMBRE, 'challenger', v_exp)`; cargá ambos con `mlflow.sklearn.load_model(f'models:/{NOMBRE}@challenger')` y guardá los F1 en `f1`; al final `client.delete_registered_model_alias(NOMBRE, 'challenger')`.

False

<details><summary>Ver solución</summary>

```python
client.set_registered_model_alias(NOMBRE, "challenger", v_exp)
f1 = {}
for alias in ["champion", "challenger"]:
    m = mlflow.sklearn.load_model(f"models:/{NOMBRE}@{alias}")
    f1[alias] = test_metrics(y_test, m.predict(X_test))["f1_macro"]
print(f1)
if f1["challenger"] > f1["champion"]:
    client.set_registered_model_alias(NOMBRE, "champion", v_exp)
client.delete_registered_model_alias(NOMBRE, "challenger")
```
(El modelo experimental, más profundo, suele perder: con estos datos gana el modelo simple.)
</details>

In [12]:
quiz(
    "La API carga models:/burnout_model_prod@champion. Entrenamos una v5 mejor. ¿Qué hay que cambiar en la API?",
    ["El código, para que cargue la v5", "Nada: solo mover el alias champion a la v5", "Reconstruir la imagen de Docker"],
    answer=1,
    explanation="Ese es el valor del Registry: el alias desacopla el código de la versión concreta.",
)

**La API carga models:/burnout_model_prod@champion. Entrenamos una v5 mejor. ¿Qué hay que cambiar en la API?**

ToggleButtons(options=('El código, para que cargue la v5', 'Nada: solo mover el alias champion a la v5', 'Reco…

Output()

## 7. Ver todo en la UI
En una terminal, desde la raíz del repo y con el entorno activado:

```bash
mlflow ui --backend-store-uri sqlite:///clases/_salidas/mlflow_clase3.db --port 5002
```

y abrir http://localhost:5002. Probá: ordenar por `f1_macro`, comparar runs del tablero, abrir el
run `busqueda_clase3` para ver los hijos, y la pestaña **Models**.

## 🏭 Casos de uso y ejemplos

### En la industria

- **Databricks** creó MLflow y lo liberó como código abierto en 2018. Hoy es la plataforma abierta
  de seguimiento más usada, y servicios como Azure ML y Databricks la ofrecen integrada.
- **Bancos y regulación.** En banca se exige documentar cada modelo de riesgo: con qué datos se
  entrenó, cómo se validó, quién lo aprobó (en EE. UU., la guía SR 11-7 de la Reserva Federal). Un
  registro de modelos con trazabilidad run → datos → métricas cubre buena parte de ese pedido.
- **Equipos grandes** usan los alias como contrato: el equipo de datos publica `@challenger`, otro
  equipo valida y mueve `@champion`. Ninguno toca el código del otro.

### Casos prácticos

**Caso 1.** Victoria dice que consiguió F1 macro 0,56 con otros hiperparámetros. ¿Cómo lo verificás
sin pedirle el notebook?

<details><summary>Respuesta</summary>

Buscás su run en MLflow: parámetros, métrica, dataset usado (`log_input`) y el commit de Git (MLflow
guarda la etiqueta `mlflow.source.git.commit`). Si el F1 es en test, revisás que no lo haya usado para
elegir. Con el código de abajo lo encontrás en una línea.
</details>

**Caso 2.** Promoviste la v3 a `champion` y al día siguiente las predicciones batch salen raras. ¿Qué
hacés primero?

<details><summary>Respuesta</summary>

**Rollback:** devolvés el alias `champion` a la v2
(`client.set_registered_model_alias("burnout_model_prod", "champion", "2")`). El DAG batch la toma en
su próxima corrida, sin cambiar código. Después investigás la v3 con calma.
</details>

**Caso 3.** Corriste el notebook del experimento y en la UI de Docker (`localhost:5001`) no aparece
nada.

<details><summary>Respuesta</summary>

El notebook probablemente usó el modo **local** (SQLite en `notebooks/_salidas/`) porque, cuando se
ejecutó la primera celda, el MLflow de Docker no respondía. Levantá el compose, reiniciá el kernel y
corré de nuevo: la primera celda tiene que decir `Modo: docker`.
</details>

**Caso 4.** Al cargar el modelo desde tu PC aparece `NoCredentialsError` o un error de conexión a S3.

<details><summary>Respuesta</summary>

Los artefactos están en MinIO, así que el cliente necesita `AWS_ACCESS_KEY_ID`,
`AWS_SECRET_ACCESS_KEY` y `MLFLOW_S3_ENDPOINT_URL=http://localhost:9000`. El notebook del experimento
los define en su primera celda.
</details>

### Ejemplos de uso de la API

```python
import mlflow
from mlflow import MlflowClient

# Los 5 mejores runs de la búsqueda, ordenados por F1 en CV
mlflow.search_runs(
    experiment_names=["Burnout Risk"],
    filter_string="tags.stage = 'optuna'",
    order_by=["metrics.cv_f1_macro DESC"],
    max_results=5,
)[["run_id", "params.max_depth", "params.learning_rate", "metrics.cv_f1_macro"]]

# ¿Qué versión es el champion hoy?
client = MlflowClient()
client.get_model_version_by_alias("burnout_model_prod", "champion").version

# Rollback: volver el champion a la versión 1
client.set_registered_model_alias("burnout_model_prod", "champion", "1")
```

Así busca el DAG `train_burnout` los mejores hiperparámetros, con `search_runs`.

## Resumen
- Run = params + métricas + tags + artefactos; los experimentos agrupan runs.
- Runs anidados para búsquedas de hiperparámetros.
- Registry: versiones + alias; producción carga siempre `@champion`.

Siguiente: **C4 — Airflow**.

## 🔁 Repaso con tarjetas
Leé el concepto, intentá explicarlo con tus palabras y recién después apretá **Mostrar respuesta**.
Marcá **La sabía** o **A repasar**: al final te queda la lista de lo que conviene volver a mirar.

In [13]:
tarjetas("C3")

## 📝 Cuestionario final
Respondé todas y apretá **Corregir**. Cada respuesta trae su explicación. Meta: 7 de 8 o más.

In [14]:
cuestionario("C3")